### Import modules

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow import keras
# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Load Data

In [ ]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

data = data.drop(columns=['file'])
data['pattern'] = data['pattern'].fillna('None Type')
data.head()

### Configure

In [ ]:
EVALUATING_ENABLED = False
TEMP_TEST_SPLIT = False

In [ ]:
if TEMP_TEST_SPLIT:
    temp_train_data,temp_test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['pattern'])
    data = temp_train_data
    temp_test_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_test_data.csv', index=False)
    temp_train_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_train_data.csv', index=False)

### NN Architecture

In [ ]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

In [ ]:
def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            # Dense(num_classes, activation="softmax"),
            Dense(num_classes, activation=None),
        ]
    )

@keras.utils.register_keras_serializable()
class TemperatureScaling(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.temperature = tf.Variable(
            initial_value=1.0,
            trainable=True,
            dtype=tf.float32,
            constraint=lambda t: tf.clip_by_value(t, 1e-6, 100.0),
        )

    def call(self, logits):
        return logits / self.temperature
    def get_config(self):
        return super().get_config()

def build_calibrated_model(base_model: tf.keras.Model) -> tf.keras.Model:
    base_model.trainable = False

    inputs = tf.keras.Input(shape=base_model.input_shape[1:])
    logits = base_model(inputs)

    scaled_logits = TemperatureScaling()(logits)
    outputs = tf.keras.layers.Softmax()(scaled_logits)

    return tf.keras.Model(inputs, outputs)

import numpy as np

def nn_train(data=data):
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)

    X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
        X,
        y_encoded,
        test_size=0.3,
        random_state=42,
        stratify=y_encoded,
    )
    X_val, X_test, y_val_enc, y_test_enc = train_test_split(
        X_temp,
        y_temp_enc,
        test_size=0.5,
        random_state=42,
        stratify=y_temp_enc,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)


    if not EVALUATING_ENABLED:
        # X_train = np.vstack([X_train, X_val])
        # y_train = np.vstack([y_train, y_val])
        X_val = np.vstack([X_val, X_test])
        y_val = np.vstack([y_val, y_test])

    def expected_calibration_error(
        probs: np.ndarray,
        y_true: np.ndarray,
        n_bins: int = 15,
    ) -> float:
        confidences = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)
        accuracies = (predictions == y_true).astype(float)

        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        N = len(y_true)

        for i in range(n_bins):
            bin_lower = bin_boundaries[i]
            bin_upper = bin_boundaries[i + 1]

            in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
            bin_size = np.sum(in_bin)

            if bin_size > 0:
                bin_accuracy = np.mean(accuracies[in_bin])
                bin_confidence = np.mean(confidences[in_bin])

                ece += (bin_size / N) * abs(bin_accuracy - bin_confidence)

        return ece


    model = build_classifier(X_train.shape[1], num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    calibrated_model = build_calibrated_model(model)

    calibrated_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
    )


    calibrated_model.fit(
        X_val,
        y_val,
        epochs=50,
        batch_size=256,
        verbose=0,
    )


    if not EVALUATING_ENABLED:
        # Persist artifacts for downstream inference pipelines
        calibrated_model.save(MODEL_PATH, include_optimizer=True)
        joblib.dump(scaler, SCALER_PATH)
        joblib.dump(label_encoder, ENCODER_PATH)
        metadata = {
            "target_column": TARGET_COLUMN,
            "numeric_features": numeric_features,
            "num_classes": num_classes,
            "label_classes": label_encoder.classes_.tolist(),
        }
        METADATA_PATH.write_text(json.dumps(metadata, indent=2))
        print(f"Saved model to {MODEL_PATH}")
        print(f"Saved scaler to {SCALER_PATH}")
        print(f"Saved label encoder to {ENCODER_PATH}")
        print(f"Saved metadata to {METADATA_PATH}")
    else:
        print("\nEvaluation enabled; not saving model artifacts.")

        test_loss, test_acc, test_top3 = calibrated_model.evaluate(X_test, y_test, verbose=0)
        print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

        y_pred = calibrated_model.predict(X_test)
        y_pred_labels = y_pred.argmax(axis=1)
        report = classification_report(
            y_test_enc,
            y_pred_labels,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        probs = calibrated_model.predict(X_test)
        ece = expected_calibration_error(probs, y_test_enc, n_bins=15)

        print(f"ECE: {ece:.4f}")


        print("\nKey metrics:")
        display(summary)
        print("\nTop classes by support:")
        display(class_breakdown)

        raw_preds = model.predict(X_test).argmax(axis=1)
        cal_preds = calibrated_model.predict(X_test).argmax(axis=1)

        print("Accuracy identical:", np.all(raw_preds == cal_preds))
    
    return calibrated_model,scaler,label_encoder

In [ ]:
nn_train(data)

### Logistic Regression classifier

In [ ]:
def lr_train(data=data):
    # Logistic regression stage intentionally skipped per latest workflow requirements.
    # The end-to-end classifier now relies solely on the neural network above.
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.25, random_state=42,stratify=y_encoded
    )

    from sklearn.linear_model import LogisticRegression

    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)

    if EVALUATING_ENABLED:
        logreg.fit(X_train, y_train)
        y_pred = logreg.predict(X_test)
        report = classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        display(summary)
    else:
        logreg.fit(X, y)
        # Save model
        ARTIFACT_DIR = Path("../models/pattern_logreg_classifier").resolve()
        ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        joblib.dump(logreg, ARTIFACT_DIR / "logistic_regression_model.joblib")
        joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
        joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
        print(f"Saved logistic regression model and artifacts to {ARTIFACT_DIR}")

    return logreg,scaler,label_encoder


In [ ]:
lr_train(data)

### Ensemble training

In [ ]:
from sklearn.model_selection import StratifiedKFold

raw_data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
raw_data['pattern'].fillna('None', inplace=True)

synthetic_data       = raw_data[raw_data['file'].str.contains('pattern',na=False)]
verified_communities = raw_data[~raw_data['file'].str.contains('pattern',na=False)]
synthetic_data = synthetic_data.drop(columns=['file'])
verified_communities = verified_communities.drop(columns=['file'])
def get_folded_splits(fold_count=4,random_state=42,use_only_verified=True,min_samples_per_class=5):
    folded_data = []

    _sd = synthetic_data.copy()
    _vd = verified_communities.copy()

    _vd_c = _vd['pattern'].value_counts()
    _vp_i = _vd_c[_vd_c>=min_samples_per_class].index.tolist()
    _vd   = _vd[_vd['pattern'].isin(_vp_i)]

    _vd_kfold = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=random_state)
    _vd_folds = _vd_kfold.split(_vd, _vd['pattern'])

    for train_indices, test_indices in _vd_folds:
        vd_train = _vd.iloc[train_indices]
        vd_test = _vd.iloc[test_indices]
        if use_only_verified:
            train_data = vd_train
            test_data  = vd_test
        else:
            train_data = pd.concat([vd_train,_sd])
            test_data  = vd_test 

        folded_data.append((train_data, test_data))
    return folded_data

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [42]:
# kfolds = StratifiedKFold(n_splits=4,random_state=42,shuffle=True)
# X = data.drop(columns=['pattern'])
# y = data['pattern']
# folds_idx = kfolds.split(X,y)
# folds=[]

# proba_dist = []

# for idx in folds_idx:
#     train,test = data.iloc[idx[0]],data.iloc[idx[1]]
#     folds.append((train,test))

folds = get_folded_splits(fold_count=3,use_only_verified=True,min_samples_per_class=12)
proba_dist = []

for fold in folds:   
    nn_model,nn_scaler,nn_le = nn_train(fold[0])
    nn_scaled = nn_scaler.transform(fold[1].drop(columns=['pattern']))
    nn_proba = pd.DataFrame(nn_model.predict(nn_scaled),columns=nn_le.classes_)
    nn_proba['pattern'] = fold[1]['pattern'].values
    
    lr_model,lr_scaler,lr_le = lr_train(fold[0])
    lr_scaled = lr_scaler.transform(fold[1].drop(columns=['pattern']))
    lr_proba = pd.DataFrame(lr_model.predict_proba(lr_scaled),columns=lr_le.classes_)
    lr_proba['pattern'] = fold[1]['pattern'].values

    numeric_cols = nn_proba.select_dtypes(include=['number']).columns.tolist()
    meta_proba = pd.DataFrame()
    meta_proba = (nn_proba[numeric_cols] + lr_proba[numeric_cols]*0.7)/2
    meta_proba['pattern'] = fold[1]['pattern'].values
    proba_dist.append({
        'train':fold[0],
        'test':fold[1],
        'nn_proba':nn_proba,
        'lr_proba':lr_proba,
        'meta_proba':meta_proba,
    })

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 368ms/step - accuracy: 0.1870 - loss: 2.9172 - top3_acc: 0.4390 - val_accuracy: 0.1852 - val_loss: 2.3700 - val_top3_acc: 0.4444 - learning_rate: 0.0010
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.3171 - loss: 2.2028 - top3_acc: 0.6504 - val_accuracy: 0.2593 - val_loss: 2.2252 - val_top3_acc: 0.5370 - learning_rate: 0.0010
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.4472 - loss: 1.7101 - top3_acc: 0.7967 - val_accuracy: 0.3148 - val_loss: 2.1028 - val_top3_acc: 0.5926 - learning_rate: 0.0010
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.6585 - loss: 1.1516 - top3_acc: 0.9187 - val_accuracy: 0.4259 - val_loss: 2.0052 - val_top3_acc: 0.6296 - learning_rate: 0.0010
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.7480 - loss: 1.1298 - top3_acc: 0.8780 - val_accuracy: 0.4074 - val_loss: 1.9235 - val_top3_acc: 0.6667 - learning_rate: 0.0010
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 330ms/step
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Epoch 1/200


...

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [43]:
dist = pd.DataFrame(columns=proba_dist[0]['meta_proba'].columns)
for fold_proba in proba_dist:
    dist = pd.concat([dist,fold_proba['meta_proba']])

/tmp/ipykernel_240734/178373797.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dist = pd.concat([dist,fold_proba['meta_proba']])


In [44]:
def get_prediction(row, class_threshold=0.5,none_threshold=0.5):
    high_prob = row.max()
    if high_prob >= class_threshold:
        max_class = row.idxmax()
        return max_class
    elif high_prob <= none_threshold:
        return 'None Type'
    return "Other"


def classify(class_threshold,none_threshold,data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction, class_threshold=class_threshold,none_threshold=none_threshold, axis=1)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

In [45]:
best = None
best_class_wise = pd.DataFrame()
for th_t in range(0,100,10):
    for thn in range(0,100,100):
        th = th_t/100
        thn=thn/100
        precision, recall, f1, report_df = classify(th,thn,dist)
        
        best_class_wise[str(th)+':'+str(thn)] = report_df['f1-score']

        if best is None or f1 > best[0]:
            best = (f1, th, precision, recall)
        print(f"Threshold: {th} => Precision: {precision}, Recall: {recall}, F1-Score: {f1}")

print("-"*10)
print(f"Best Threshold: {best[1]} => Precision: {best[2]}, Recall: {best[3]}, F1-Score: {best[0]}")
best_class_wise['best_threshold'] = best_class_wise.idxmax(axis=1)

Threshold: 0.0 => Precision: 0.47968478650848567, Recall: 0.4340010454150526, F1-Score: 0.44549740614699496
Threshold: 0.1 => Precision: 0.47968478650848567, Recall: 0.4340010454150526, F1-Score: 0.44549740614699496
Threshold: 0.2 => Precision: 0.4364877330242442, Recall: 0.3906009408735473, F1-Score: 0.40291535906078746
Threshold: 0.3 => Precision: 0.47878080772374, Recall: 0.31554460831721476, F1-Score: 0.3592642317718796
Threshold: 0.4 => Precision: 0.5732957393483711, Recall: 0.2581345334137887, F1-Score: 0.3373825322457783
Threshold: 0.5 => Precision: 0.5869696969696969, Recall: 0.158005308816479, F1-Score: 0.23174977951236597
Threshold: 0.6 => Precision: 0.27999999999999997, Recall: 0.06682888624377988, F1-Score: 0.10713374165691032
Threshold: 0.7 => Precision: 0.35583333333333333, Recall: 0.034539949067874595, F1-Score: 0.06236801774551322
Threshold: 0.8 => Precision: 0.06666666666666667, Recall: 0.006060606060606061, F1-Score: 0.01111111111111111
Threshold: 0.9 => Precision: 0.

In [46]:
best_class_wise

,0.0:0.0,0.1:0.0,0.2:0.0,0.3:0.0,0.4:0.0,0.5:0.0,0.6:0.0,0.7:0.0,0.8:0.0,0.9:0.0,best_threshold
Classical Models,0.685714,0.685714,0.685714,0.720000,0.697674,0.563380,0.354839,0.120000,0.000000,0.0,0.3:0.0
LLM Results Evaluation,0.181818,0.181818,0.190476,0.117647,0.125000,0.000000,0.000000,0.000000,0.000000,0.0,0.2:0.0
LLM based Multimodal Generative Prompting,0.520000,0.520000,0.520000,0.478261,0.473684,0.388889,0.129032,0.068966,0.000000,0.0,0.0:0.0
Model Abstraction Pattern,0.434783,0.434783,0.434783,0.400000,0.315789,0.117647,0.000000,0.000000,0.000000,0.0,0.0:0.0
Modular LLM Agent Architectures,0.432432,0.432432,0.432432,0.387097,0.320000,0.095238,0.000000,0.000000,0.000000,0.0,0.0:0.0
None,0.496241,0.496241,0.500000,0.464000,0.427184,0.325581,0.210526,0.138889,0.000000,0.0,0.2:0.0
Preprocessing Text and Numerical Data,0.358209,0.358209,0.358209,0.327869,0.296296,0.279070,0.195122,0.157895,0.111111,0.0,0.0:0.0
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.486486,0.486486,0.486486,0.344828,0.370370,0.240000,0.000000,0.000000,0.000000,0.0,0.0:0.0
Tool Use for LLMs,0.413793,0.413793,0.421053,0.352941,0.347826,0.307692,0.181818,0.137931,0.000000,0.0,0.2:0.0
accuracy,0.488722,0.488722,0.488722,0.417293,0.338346,0.218045,0.105263,0.052632,0.007519,0.0,0.0:0.0


In [47]:
best_df = pd.DataFrame()
best_df['Class Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[0])
best_df['None Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[1])

In [48]:
best_df

,Class Threshold,None Threshold
Classical Models,0.3,0.0
LLM Results Evaluation,0.2,0.0
LLM based Multimodal Generative Prompting,0.0,0.0
Model Abstraction Pattern,0.0,0.0
Modular LLM Agent Architectures,0.0,0.0
None,0.2,0.0
Preprocessing Text and Numerical Data,0.0,0.0
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.0,0.0
Tool Use for LLMs,0.2,0.0
accuracy,0.0,0.0


In [53]:
def get_prediction_trained(row):
    high_prob = row.max()
    max_class = row.idxmax()
    if float(best_df['Class Threshold'][max_class]) <= float(high_prob):
        return max_class
    return "Other"


def classify_trained(data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_trained, axis=1)
    print(f"pricision:{precision} recall:{recall} f1:{f1}")
    display(y_pred.value_counts())
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained(dist)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
with_threshold_report = report_df.T

pricision:0.44234329580888965 recall:0.3906009408735473 f1:0.4063439304893589


None                                                         68
Classical Models                                             53
Preprocessing Text and Numerical Data                        34
Tool Use for LLMs                                            30
LLM based Multimodal Generative Prompting                    24
Modular LLM Agent Architectures                              17
Retrieval Augmented Generation(RAG) Optimization for LLMs    17
LLM Results Evaluation                                        8
Other                                                         8
Model Abstraction Pattern                                     7
Name: count, dtype: int64

Precision: 0.44234329580888965, Recall: 0.3906009408735473, F1-Score: 0.4063439304893589


In [50]:
report_df


,precision,recall,f1-score,support
Classical Models,0.679245,0.765957,0.720000,47.000000
LLM Results Evaluation,0.250000,0.153846,0.190476,13.000000
LLM based Multimodal Generative Prompting,0.541667,0.500000,0.520000,26.000000
Model Abstraction Pattern,0.714286,0.312500,0.434783,16.000000
Modular LLM Agent Architectures,0.470588,0.400000,0.432432,20.000000
None,0.485294,0.515625,0.500000,64.000000
Other,0.000000,0.000000,0.000000,0.000000
Preprocessing Text and Numerical Data,0.352941,0.363636,0.358209,33.000000
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.529412,0.450000,0.486486,20.000000
Tool Use for LLMs,0.400000,0.444444,0.421053,27.000000


In [35]:
def get_prediction_no_threshold(row):
    max_class = row.idxmax()
    return max_class


def classify_trained_no_threshold(data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_no_threshold, axis=1)
    print(y_true)
    print(y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained_no_threshold(dist)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
without_threshold_report = report_df.T

0      Classical Models
1      Classical Models
2      Classical Models
3      Classical Models
4      Classical Models
            ...        
83    Tool Use for LLMs
84    Tool Use for LLMs
85    Tool Use for LLMs
86    Tool Use for LLMs
87    Tool Use for LLMs
Name: pattern, Length: 266, dtype: object
0                          Classical Models
1                          Classical Models
2                          Classical Models
3                          Classical Models
4                                      None
                      ...                  
83                        Tool Use for LLMs
84                   LLM Results Evaluation
85                        Tool Use for LLMs
86                Model Abstraction Pattern
87    Preprocessing Text and Numerical Data
Length: 266, dtype: object
Precision: 0.26456043084323383, Recall: 0.23224951669931723, F1-Score: 0.24353672148537553


In [36]:
report_df

,precision,recall,f1-score,support
Classical Models,0.622642,0.702128,0.660000,47.000000
Explainable AI (XAI) Techniques,0.000000,0.000000,0.000000,0.000000
Integrating External Knlowladge with LLM,0.000000,0.000000,0.000000,0.000000
LLM Code Execution for Precision,0.000000,0.000000,0.000000,0.000000
LLM Context Management,0.000000,0.000000,0.000000,0.000000
LLM Results Evaluation,0.285714,0.153846,0.200000,13.000000
LLM based Multimodal Generative Prompting,0.640000,0.615385,0.627451,26.000000
"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",0.000000,0.000000,0.000000,0.000000
LLMs for Recommender Systems,0.000000,0.000000,0.000000,0.000000
Model Abstraction Pattern,0.636364,0.437500,0.518519,16.000000


In [ ]:
with_threshold_report.rename(index={
    'precision':'precision (with thresholds)',
    'recall':'recall (with thresholds)',
    'f1-score':'f1-score (with thresholds)'
},inplace=True)

without_threshold_report.rename(index={
    'precision':'precision (without thresholds)',
    'recall':'recall (without thresholds)',
    'f1-score':'f1-score (without thresholds)'
},inplace=True)

final_report = pd.concat([with_threshold_report,without_threshold_report])

In [ ]:
final_report.to_csv('result/scores/threshold_report.csv')

### Second level Model

In [56]:
proba_dist[0]['meta_proba']

,Classical Models,LLM Results Evaluation,LLM based Multimodal Generative Prompting,Model Abstraction Pattern,Modular LLM Agent Architectures,None,Preprocessing Text and Numerical Data,Retrieval Augmented Generation(RAG) Optimization for LLMs,Tool Use for LLMs,pattern
0,0.486537,0.019424,0.092743,0.084568,0.028804,0.055912,0.035376,0.027764,0.018873,Classical Models
1,0.466435,0.026395,0.083679,0.085204,0.038456,0.022245,0.056822,0.042929,0.027836,Classical Models
2,0.325771,0.042608,0.060028,0.050407,0.047151,0.191695,0.077711,0.035997,0.018632,Classical Models
3,0.496008,0.046264,0.072080,0.054814,0.041552,0.030182,0.043517,0.048792,0.016792,Classical Models
4,0.084391,0.065737,0.053391,0.077073,0.080232,0.297869,0.114901,0.033112,0.043294,Classical Models
...,...,...,...,...,...,...,...,...,...,...
84,0.112199,0.038173,0.051439,0.049248,0.037917,0.452551,0.011838,0.072205,0.024432,Tool Use for LLMs
85,0.016517,0.060508,0.014473,0.023566,0.305277,0.205260,0.078934,0.024177,0.121289,Tool Use for LLMs
86,0.009413,0.008205,0.007594,0.050944,0.306201,0.061834,0.010218,0.026595,0.368995,Tool Use for LLMs
87,0.052379,0.046155,0.305536,0.050454,0.204946,0.041654,0.069209,0.041305,0.038363,Tool Use for LLMs


In [57]:
from sklearn.linear_model import LogisticRegression

lr_pred = pd.DataFrame(columns=['prediction','pattern'])
c=0
for fold in proba_dist:
    print("Processing fold:",c)
    X_meta = fold['meta_proba'].drop(columns=['pattern'])
    X_lr = fold['lr_proba'].drop(columns=['pattern'])
    X_nn = fold['nn_proba'].drop(columns=['pattern'])
    y_temp = fold['meta_proba']['pattern']

    X_meta = X_meta.add_prefix('meta_')
    X_lr = X_lr.add_prefix('lr_')
    X_nn = X_nn.add_prefix('nn_')

    dataset = pd.concat([X_meta,X_nn,X_lr,y_temp], axis=1)

    # display(dataset)

    lr_sec_model,le_sec_scaler,le_sec_encoder = lr_train(dataset)
    lr_sec_scaled = le_sec_scaler.transform(dataset.drop(columns=['pattern']))
    lr_sec_pred = pd.DataFrame(columns=['prediction','pattern'])
    lr_sec_pred['prediction'] = lr_sec_model.predict(lr_sec_scaled)
    lr_sec_pred['pattern'] = dataset['pattern'].values
    lr_pred = pd.concat([lr_pred,lr_sec_pred])
    c+=1

Processing fold: 0
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 1


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 2


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [58]:
dataset

,meta_Classical Models,meta_LLM Results Evaluation,meta_LLM based Multimodal Generative Prompting,meta_Model Abstraction Pattern,meta_Modular LLM Agent Architectures,meta_None,meta_Preprocessing Text and Numerical Data,meta_Retrieval Augmented Generation(RAG) Optimization for LLMs,meta_Tool Use for LLMs,nn_Classical Models,...,lr_Classical Models,lr_LLM Results Evaluation,lr_LLM based Multimodal Generative Prompting,lr_Model Abstraction Pattern,lr_Modular LLM Agent Architectures,lr_None,lr_Preprocessing Text and Numerical Data,lr_Retrieval Augmented Generation(RAG) Optimization for LLMs,lr_Tool Use for LLMs,pattern
0,0.545220,0.023754,0.038010,0.017482,0.020765,0.079077,0.041154,0.044148,0.040390,0.391956,...,0.997834,0.000017,0.000120,0.000022,0.000030,0.001903,0.000032,0.000024,0.000018,Classical Models
1,0.429580,0.066642,0.036239,0.029139,0.032903,0.110198,0.037174,0.059826,0.048299,0.310568,...,0.783703,0.012683,0.004719,0.001156,0.000728,0.191890,0.003453,0.000828,0.000841,Classical Models
2,0.310236,0.137844,0.221611,0.029045,0.026975,0.002283,0.029216,0.008869,0.083921,0.241972,...,0.540714,0.249489,0.149096,0.050171,0.005925,0.000081,0.003348,0.000920,0.000256,Classical Models
3,0.205520,0.078338,0.189993,0.022043,0.031604,0.026251,0.232900,0.028885,0.034466,0.330578,...,0.114946,0.024784,0.442955,0.002316,0.000705,0.020379,0.392030,0.000729,0.001157,Classical Models
4,0.466304,0.062407,0.136116,0.026066,0.027440,0.012441,0.061975,0.025859,0.031391,0.355719,...,0.824128,0.002033,0.115606,0.000625,0.000169,0.000878,0.055298,0.001151,0.000112,Classical Models
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,0.016872,0.027255,0.025028,0.019772,0.031115,0.281078,0.116032,0.045961,0.286887,0.032760,...,0.001406,0.004987,0.016644,0.000423,0.002007,0.248712,0.006082,0.029410,0.690329,Tool Use for LLMs
84,0.038702,0.033327,0.007215,0.062445,0.017305,0.062061,0.024729,0.067249,0.536967,0.055076,...,0.031899,0.010645,0.000407,0.000635,0.000314,0.001960,0.000553,0.004138,0.949450,Tool Use for LLMs
85,0.051140,0.066299,0.018888,0.133959,0.035640,0.043665,0.040857,0.054179,0.405373,0.088106,...,0.020249,0.000182,0.002440,0.210527,0.000891,0.000295,0.022261,0.001674,0.741481,Tool Use for LLMs
86,0.020827,0.056170,0.017264,0.293275,0.083829,0.011476,0.047060,0.067647,0.252451,0.039823,...,0.002615,0.010237,0.014162,0.635026,0.106450,0.013255,0.065718,0.096448,0.056088,Tool Use for LLMs


In [38]:
report = classification_report(lr_pred['prediction'], lr_pred['pattern'], output_dict=True, zero_division=0)
pd.DataFrame(report).T

,precision,recall,f1-score,support
Classical Models,0.914894,0.811321,0.860000,53.000000
LLM Results Evaluation,0.615385,1.000000,0.761905,8.000000
LLM based Multimodal Generative Prompting,0.884615,0.884615,0.884615,26.000000
Model Abstraction Pattern,0.875000,1.000000,0.933333,14.000000
Modular LLM Agent Architectures,0.800000,0.842105,0.820513,19.000000
None,0.906250,0.773333,0.834532,75.000000
Preprocessing Text and Numerical Data,0.818182,0.931034,0.870968,29.000000
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.850000,1.000000,0.918919,17.000000
Tool Use for LLMs,0.740741,0.800000,0.769231,25.000000
accuracy,0.849624,0.849624,0.849624,0.849624
